# Perfiles topográficos transversales a fallas

## Objetivo

Caracterizar la expresión topográfica de las fallas cartografiadas del área
de estudio mediante perfiles transversales generados automáticamente.

Se analizarán las trazas de Carranza, Pelluhue y El Yolki, manteniendo
separados los registros de la capa y los sectores de cada traza.

Buscamos reconocer cambios de pendiente y contrastes de relieve que
permitan seleccionar sectores para una revisión geomorfológica detallada
y una posterior comparación de los drenajes de ambos bloques.

## Alcance y limitaciones

- Los perfiles representan la topografía actual, no directamente el
  desplazamiento de las fallas.
- Un desnivel puede incluir efectos de erosión, incisión fluvial,
  diferencias litológicas y relieve previo.
- La posición cero corresponde a la traza cartografiada; no garantiza
  la ubicación exacta de una ruptura o del pie de un escarpe.
- La capa contiene rumbo, sentido de buzamiento y cinemática, pero no
  informa el ángulo de buzamiento.
- En las fallas normal–sinistrales, los cortes topográficos no permiten
  cuantificar la componente lateral del movimiento.
- Los resultados son exploratorios y requieren revisión espacial.

## 1. Directorios y parámetros

Definimos las rutas del DEM original, la capa de fallas y la carpeta
de salida. Este notebook es independiente de los cálculos de χ y ksn.

La configuración inicial utiliza:

| Parámetro | Valor inicial | Función |
|---|---:|---|
| Longitud objetivo del sector | 2000 m | Dividir las trazas para no mezclar todo su relieve |
| Extensión transversal | 1000 m por lado | Delimitar el terreno examinado a ambos lados |
| Separación objetivo entre transectas | 100 m | Distribuir los perfiles a lo largo de la falla |
| Ventana de orientación local | 200 m | Estimar la dirección local de la traza |
| Mínimo de perfiles válidos | 3 | Permitir el cálculo del resumen transversal |

Estos valores son exploratorios. La longitud de los sectores y la separación
real entre transectas se ajustan para distribuirlas uniformemente.

Si cambiamos estos parámetros, debemos repetir la generación de perfiles.
Cambiar únicamente los sectores que se muestran no requiere recalcularlos.

In [ ]:
import sys
import importlib
import faulthandler

faulthandler.enable()

print("Python:", sys.version, flush=True)
print("Intérprete:", sys.executable, flush=True)

for nombre in ["numpy", "pandas", "shapely", "geopandas", "matplotlib.pyplot"]:
    print(f"Importando {nombre}...", flush=True)
    modulo = importlib.import_module(nombre)
    print(f"OK: {nombre}", flush=True)

print("Importaciones completadas. No se cargaron datos.", flush=True)

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

from shapely.geometry import LineString
from IPython.display import display

BASE = Path(
    r"C:\datos"
    r"\Memoria\Codigos Stanford"
)

RUTA_DEM = BASE.parent / "TDM_cut_UTM18.tif"

RUTA_FALLAS = Path(
    r"C:\datos\capas_externas\Nueva carpeta"
    r"\fallas_CHAF_area_estudio.shp"
)

RUTA_SALIDA = BASE / "Productos" / "Perfiles_y_fallas"

# ── Parámetros exploratorios ──────────────────────────────────────────
LONGITUD_SECTOR_M = 2000.0
DISTANCIA_POR_LADO_M = 1000.0
SEPARACION_TRANSECTAS_M = 100.0
VENTANA_RUMBO_LOCAL_M = 200.0

# Número mínimo de perfiles con elevación válida para resumir
# una posición transversal.
MIN_PERFILES_VALIDOS = 3

# Umbral geométrico: evita asignar bloques cuando la perpendicular
# local es poco compatible con el sentido de buzamiento registrado.
MIN_ALINEACION_BUZAMIENTO = 0.5

for ruta in (RUTA_DEM, RUTA_FALLAS):
    if not ruta.exists():
        raise FileNotFoundError(ruta)

for valor in (
    LONGITUD_SECTOR_M,
    DISTANCIA_POR_LADO_M,
    SEPARACION_TRANSECTAS_M,
    VENTANA_RUMBO_LOCAL_M,
):
    if not np.isfinite(valor) or valor <= 0:
        raise ValueError("Las distancias deben ser positivas y finitas.")

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "font.size": 10,
    "savefig.dpi": 200,
})

print("Configuración preparada.")

## 2. Carga y comprobación espacial

Cargamos las fallas y verificamos el sistema de coordenadas del DEM:
**WGS 84 / UTM zona 18S — EPSG:32718**.

Las elevaciones se obtienen del DEM original, no de una superficie
rellenada o acondicionada para calcular direcciones de flujo.

Comprobamos:

- Existencia de los archivos.
- Sistema de coordenadas.
- Atributos requeridos.
- Geometrías lineales válidas.
- Disponibilidad de información de buzamiento.

Las geometrías multipartes se separan, conservando su identificador
de origen. No se fusionan registros diferentes de una misma falla.

### Convención de los perfiles

- **Distancia 0:** intersección con la traza cartografiada.
- **Distancias negativas:** lado hacia el que buza la falla,
  correspondiente al bloque colgante según los atributos.
- **Distancias positivas:** lado opuesto, correspondiente al bloque yaciente.

Esta asignación depende de la información estructural, no de cuál lado
tiene mayor elevación actual.

In [ ]:
with rasterio.open(RUTA_DEM) as src:
    CRS_DEM = src.crs
    RESOLUCION_M = max(abs(src.res[0]), abs(src.res[1]))
    LIMITES_DEM = src.bounds

if CRS_DEM is None or CRS_DEM.to_epsg() != 32718:
    raise ValueError("Se esperaba el DEM en EPSG:32718.")

fallas = gpd.read_file(RUTA_FALLAS)

if fallas.crs is None:
    raise ValueError("La capa de fallas no tiene CRS.")

fallas = fallas.to_crs(CRS_DEM)

requeridas = {"f_id", "falla", "rumbo", "buz", "dir_buz", "sentido"}
faltantes = requeridas - set(fallas.columns)
if faltantes:
    raise ValueError(f"Faltan atributos: {faltantes}")

if (
    fallas.geometry.isna().any()
    or fallas.geometry.is_empty.any()
    or (~fallas.geometry.is_valid).any()
):
    raise ValueError("Hay geometrías ausentes, vacías o inválidas.")

if not fallas.geom_type.isin(["LineString", "MultiLineString"]).all():
    raise ValueError("La capa debe contener únicamente líneas.")

# No unir registros distintos de una misma falla.
fallas = fallas.explode(index_parts=False).reset_index(drop=True)
fallas["registro"] = np.arange(1, len(fallas) + 1)
fallas["f_id"] = fallas["f_id"].astype(str)

# Muestreo transversal cercano a la resolución del DEM.
# Linspace garantiza incluir exactamente el cero.
n_pasos = int(np.ceil(DISTANCIA_POR_LADO_M / RESOLUCION_M))
DISTANCIAS = np.linspace(
    -DISTANCIA_POR_LADO_M,
    DISTANCIA_POR_LADO_M,
    2 * n_pasos + 1,
)

print(f"Partes de trazas: {len(fallas)}")
print(f"Resolución del DEM: {RESOLUCION_M:g} m")
print(f"Ángulos de buzamiento informados: {fallas['buz'].notna().sum()}")
print("Distancia negativa: lado del buzamiento / bloque colgante.")
print("Distancia positiva: lado opuesto / bloque yaciente.")

display(fallas[
    ["registro", "f_id", "falla", "rumbo", "buz", "dir_buz", "sentido"]
])

## 3. Generación automática y extracción de elevaciones

Cada traza se divide en sectores de longitud similar. Dentro de cada sector:

1. Distribuimos puntos a lo largo de la falla.
2. Estimamos la orientación local de la traza.
3. Construimos transectas perpendiculares.
4. Orientamos los cortes según el sentido de buzamiento registrado.
5. Extraemos las elevaciones del DEM a intervalos próximos a su resolución.
6. Resumimos las elevaciones a igual distancia transversal de la falla.

Se utiliza el valor de la celda del DEM, sin interpolar ni rellenar
datos ausentes.

### Control de orientación y cobertura

Las transectas con orientación ambigua respecto al sentido de buzamiento
se omiten y se registran como incidencias.

También se conserva la diferencia entre el rumbo local calculado y el
rumbo del atributo para facilitar su revisión. Una diferencia no implica
necesariamente un error: la traza puede cambiar de orientación.

Las posiciones fuera del DEM o sin datos válidos permanecen vacías.
El resumen solo se calcula donde existe el mínimo requerido de perfiles.

### Significado del resumen

Para cada distancia transversal se calculan:

- Mediana de elevación.
- Percentil 10.
- Percentil 90.
- Número de perfiles con datos válidos.

Este procedimiento resume transectas alineadas en la falla. No corresponde
a un promedio de todos los píxeles de un buffer ni a observaciones
estadísticamente independientes.

En sectores curvos, las transectas pueden acercarse o cruzarse y muestrear
terreno repetido. Su geometría debe revisarse antes de interpretar el corte.

In [ ]:
AZIMUT_BUZAMIENTO = {
    "N": 0, "NE": 45, "E": 90, "SE": 135,
    "S": 180, "SW": 225, "W": 270, "NW": 315,
}

resultados = {}
registros_transectas = []
registros_sectores = []
incidencias = []

with rasterio.open(RUTA_DEM) as dem:
    for _, falla in fallas.iterrows():

        linea = falla.geometry
        longitud = linea.length
        registro = int(falla["registro"])
        fid = falla["f_id"]

        direccion = str(falla["dir_buz"]).strip().upper()

        if direccion not in AZIMUT_BUZAMIENTO:
            incidencias.append({
                "f_id": fid,
                "sector": None,
                "motivo": "Sentido de buzamiento no reconocido",
            })
            continue

        if longitud <= 0:
            incidencias.append({
                "f_id": fid,
                "sector": None,
                "motivo": "Longitud nula",
            })
            continue

        azimut = np.radians(AZIMUT_BUZAMIENTO[direccion])
        vector_buz = np.array([np.sin(azimut), np.cos(azimut)])

        # Dividir en sectores similares, evitando un último sector diminuto.
        nsectores = max(1, int(np.ceil(longitud / LONGITUD_SECTOR_M)))
        limites = np.linspace(0, longitud, nsectores + 1)

        for numero in range(nsectores):
            inicio, fin = limites[numero:numero + 2]

            clave = f"{fid}_R{registro:02d}_S{numero + 1:02d}"

            # Muestreo en puntos medios: evita repetir límites de sectores.
            cantidad = max(
                1,
                int(np.ceil((fin - inicio) / SEPARACION_TRANSECTAS_M))
            )
            posiciones = (
                inicio
                + (np.arange(cantidad) + 0.5) * (fin - inicio) / cantidad
            )

            perfiles_z = []

            for numero_t, posicion in enumerate(posiciones, start=1):
                mitad = VENTANA_RUMBO_LOCAL_M / 2

                p0 = linea.interpolate(max(0, posicion - mitad))
                p1 = linea.interpolate(min(longitud, posicion + mitad))
                centro = linea.interpolate(posicion)

                tangente = np.array([p1.x - p0.x, p1.y - p0.y])
                norma = np.linalg.norm(tangente)

                if norma == 0:
                    incidencias.append({
                        "f_id": fid, "sector": clave,
                        "motivo": "No se pudo definir la orientación local",
                    })
                    continue

                tangente /= norma
                normal = np.array([-tangente[1], tangente[0]])

                # Normal positiva hacia el buzamiento.
                if np.dot(normal, vector_buz) < 0:
                    normal *= -1

                alineacion = float(np.dot(normal, vector_buz))

                if alineacion < MIN_ALINEACION_BUZAMIENTO:
                    incidencias.append({
                        "f_id": fid, "sector": clave,
                        "motivo": "Orientación local ambigua respecto al buzamiento",
                    })
                    continue

                # Distancias negativas hacia el buzamiento:
                # izquierda del gráfico = bloque colgante.
                xy = (
                    np.array([centro.x, centro.y])[None, :]
                    - DISTANCIAS[:, None] * normal[None, :]
                )

                # Muestreo de celda del DEM, sin interpolar ni rellenar vacíos.
                 # Lectura directa por ventanas, sin dem.sample.
                # Conserva el valor de celda, sin interpolar.
                from rasterio.windows import Window

                z = np.full(len(xy), np.nan, dtype=float)

                # Coordenadas UTM -> posiciones de píxel.
                inversa = ~dem.transform
                columnas_float = (
                    inversa.a * xy[:, 0]
                    + inversa.b * xy[:, 1]
                    + inversa.c
                )
                filas_float = (
                    inversa.d * xy[:, 0]
                    + inversa.e * xy[:, 1]
                    + inversa.f
                )

                dentro = (
                    np.isfinite(columnas_float)
                    & np.isfinite(filas_float)
                    & (columnas_float >= 0)
                    & (columnas_float < dem.width)
                    & (filas_float >= 0)
                    & (filas_float < dem.height)
                )

                indices = np.flatnonzero(dentro)

                if len(indices):
                    columnas = np.floor(
                        columnas_float[indices]
                    ).astype(np.int64)

                    filas = np.floor(
                        filas_float[indices]
                    ).astype(np.int64)

                    # Agrupar puntos en ventanas pequeñas de 256 × 256.
                    # Evita cargar todo el DEM o una ventana muy grande.
                    tamano = 256
                    bloques = np.column_stack([
                        filas // tamano,
                        columnas // tamano,
                    ])

                    bloques_unicos, grupo = np.unique(
                        bloques, axis=0, return_inverse=True
                    )

                    for numero_bloque, (bloque_f, bloque_c) in enumerate(
                        bloques_unicos
                    ):
                        seleccion = grupo == numero_bloque

                        fila_inicio = int(bloque_f * tamano)
                        columna_inicio = int(bloque_c * tamano)

                        alto = min(tamano, dem.height - fila_inicio)
                        ancho = min(tamano, dem.width - columna_inicio)

                        ventana = dem.read(
                            1,
                            window=Window(
                                columna_inicio,
                                fila_inicio,
                                ancho,
                                alto,
                            ),
                            masked=True,
                        )

                        valores = ventana[
                            filas[seleccion] - fila_inicio,
                            columnas[seleccion] - columna_inicio,
                        ]

                        z[indices[seleccion]] = (
                            np.ma.asarray(valores, dtype=float)
                            .filled(np.nan)
                        )

                z[~np.isfinite(z)] = np.nan
                perfiles_z.append(z)

                rumbo_local = float(
                    np.degrees(np.arctan2(tangente[0], tangente[1])) % 180
                )
                rumbo_atributo = pd.to_numeric(
                    falla["rumbo"], errors="coerce"
                )
                diferencia = (
                    abs((rumbo_local - rumbo_atributo + 90) % 180 - 90)
                    if pd.notna(rumbo_atributo) else np.nan
                )

                registros_transectas.append({
                    "sector": clave,
                    "transecta": numero_t,
                    "f_id": fid,
                    "falla": falla["falla"],
                    "dir_buz": direccion,
                    "cinematica": falla["sentido"],
                    "rumbo_loc": rumbo_local,
                    "dif_rumbo": diferencia,
                    "alineacion": alineacion,
                    "fr_valid": float(np.isfinite(z).mean()),
                    "geometry": LineString(xy),
                })

            if not perfiles_z:
                incidencias.append({
                    "f_id": fid, "sector": clave,
                    "motivo": "Sector sin transectas utilizables",
                })
                continue

            matriz = np.vstack(perfiles_z)
            nvalidos = np.isfinite(matriz).sum(axis=0)

            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                p10, mediana, p90 = np.nanpercentile(
                    matriz, [10, 50, 90], axis=0
                )

            suficiente = nvalidos >= MIN_PERFILES_VALIDOS
            p10[~suficiente] = np.nan
            mediana[~suficiente] = np.nan
            p90[~suficiente] = np.nan

            tabla = pd.DataFrame({
                "distancia_m": DISTANCIAS,
                "p10_m": p10,
                "mediana_m": mediana,
                "p90_m": p90,
                "n_validos": nvalidos,
            })

            resultados[clave] = {
                "f_id": fid,
                "falla": falla["falla"],
                "tipo": falla.get("tipo", ""),
                "cinematica": falla["sentido"],
                "dir_buz": direccion,
                "z": matriz,
                "tabla": tabla,
            }

            registros_sectores.append({
                "sector": clave,
                "f_id": fid,
                "falla": falla["falla"],
                "tipo": falla.get("tipo", ""),
                "cinematica": falla["sentido"],
                "inicio_m": inicio,
                "fin_m": fin,
                "n_perfiles": len(matriz),
                "fraccion_valida": float(np.isfinite(matriz).mean()),
                "fraccion_resumible": float(suficiente.mean()),
            })

        print(f"Revisada: {fid} — {falla['falla']}")

if not resultados:
    raise ValueError("No se generaron perfiles. Revisa las incidencias.")

transectas = gpd.GeoDataFrame(
    registros_transectas, geometry="geometry", crs=CRS_DEM
)
resumen = pd.DataFrame(registros_sectores)
incidencias_df = pd.DataFrame(incidencias)

print(f"\nSectores procesados: {len(resultados)}")
print(f"Transectas generadas: {len(transectas)}")
print(f"Incidencias: {len(incidencias_df)}")

display(resumen)

if not incidencias_df.empty:
    display(incidencias_df)

In [ ]:
from pathlib import Path
import faulthandler

faulthandler.enable()

ruta_dem_prueba = Path(
    r"C:\datos"
    r"\Memoria\TDM_cut_UTM18.tif"
)

print("1. Importando rasterio...", flush=True)
import rasterio
from rasterio.windows import Window

print("2. Abriendo el DEM...", flush=True)

with rasterio.open(ruta_dem_prueba) as dem:
    print(
        f"DEM abierto: {dem.width} × {dem.height} píxeles\n"
        f"CRS: {dem.crs}",
        flush=True,
    )

    print("3. Leyendo una ventana pequeña...", flush=True)

    ancho = min(128, dem.width)
    alto = min(128, dem.height)
    columna = (dem.width - ancho) // 2
    fila = (dem.height - alto) // 2

    muestra = dem.read(
        1,
        window=Window(columna, fila, ancho, alto),
        masked=True,
    )

    print(
        f"Lectura correcta: {muestra.shape}; "
        f"píxeles válidos: {muestra.count()}",
        flush=True,
    )

    print("4. Probando dem.sample con un punto...", flush=True)

    x, y = dem.xy(dem.height // 2, dem.width // 2)
    valor = next(dem.sample([(x, y)], indexes=1, masked=True))

    print("Muestreo correcto:", valor, flush=True)

print("5. Prueba terminada; DEM cerrado.", flush=True)

In [ ]:
from pathlib import Path
import faulthandler
import rasterio
from rasterio.windows import Window

faulthandler.enable()

ruta_dem_prueba = Path(
    r"C:\datos"
    r"\Memoria\TDM_cut_UTM18.tif"
)

print("1. Abriendo DEM...", flush=True)

with rasterio.open(ruta_dem_prueba) as dem:
    fila = dem.height // 2
    columna = dem.width // 2

    print("2. Leyendo una celda directamente, sin dem.sample...", flush=True)

    dato = dem.read(
        1,
        window=Window(columna, fila, 1, 1),
        masked=True,
    )

    print("Lectura correcta. Valor:", dato, flush=True)

print("3. DEM cerrado. Prueba terminada.", flush=True)

## 4. Visualización y revisión geomorfológica

Seleccionamos los sectores que queremos visualizar sin repetir
la extracción de elevaciones.

Cada figura contiene:

- **Mapa de control:** traza de falla, transectas y extremos situados
  hacia el sentido de buzamiento.
- **Corte transversal:** perfiles individuales, mediana y banda P10–P90.

El mapa permite comprobar la distribución geométrica, pero no incluye
el relieve de fondo. La capa de transectas puede revisarse sobre el DEM
en QGIS.

La banda P10–P90 representa variabilidad topográfica entre perfiles:
**no es un intervalo de confianza ni una estimación del error del DEM**.

### Preguntas para revisar cada sector

- ¿El cambio de pendiente aparece en varios perfiles o solo en uno?
- ¿Se encuentra cerca de la traza cartografiada?
- ¿Los perfiles atraviesan valles, divisorias o superficies comparables?
- ¿Hay suficiente cobertura de datos a ambos lados?
- ¿La curvatura de la falla produce cruces entre transectas?
- ¿La traza está clasificada como observada o inferida?

Un contraste de elevación no demuestra por sí solo un escarpe tectónico.
La ausencia de una expresión clara también debe registrarse.

No se calcula todavía una altura de escarpe, un desplazamiento
ni una tasa de movimiento.

In [ ]:
# ── VISUALIZACIÓN: solo cortes, con orientación geográfica ────────────

SECTORES_VER = (
    resumen.groupby("f_id", sort=False)["sector"].first().tolist()
)

# Para elegir sectores concretos:
# SECTORES_VER = ["300904_R03_S01", "300903_R02_S01"]

MOSTRAR_INDIVIDUALES = True
MOSTRAR_BANDA = False


def nombre_direccion(vector):
    """Dirección aproximada de un vector Este–Norte."""
    azimut = np.degrees(np.arctan2(vector[0], vector[1])) % 360

    nombres = [
        "N · Norte", "NE · Noreste",
        "E · Este", "SE · Sureste",
        "S · Sur", "SW · Suroeste",
        "W · Oeste", "NW · Noroeste",
    ]
    return nombres[int(np.floor((azimut + 22.5) / 45)) % 8]


def figura_sector(clave):
    r = resultados[clave]
    tabla = r["tabla"]
    t = transectas[transectas["sector"] == clave]

    if t.empty:
        raise ValueError(f"No hay transectas para {clave}.")

    # Vector de cada transecta desde su extremo negativo al positivo.
    vectores = np.array([
        np.asarray(g.coords[-1]) - np.asarray(g.coords[0])
        for g in t.geometry
    ], dtype=float)

    vectores /= np.linalg.norm(vectores, axis=1)[:, None]
    vector_medio = vectores.mean(axis=0)

    norma = np.linalg.norm(vector_medio)
    if norma < 0.5:
        raise ValueError(
            f"{clave}: orientaciones demasiado variables. "
            "Conviene revisar o reducir el sector."
        )

    vector_medio /= norma

    # Queremos el extremo más al norte a la izquierda.
    # Para cortes prácticamente E–W, colocamos el oeste a la izquierda.
    invertir = (
        vector_medio[1] > 0
        if abs(vector_medio[1]) > 1e-6
        else vector_medio[0] < 0
    )

    signo = -1 if invertir else 1

    x_original = tabla["distancia_m"].to_numpy()
    x_grafico = signo * x_original
    orden = np.argsort(x_grafico)
    x = x_grafico[orden]

    z = r["z"][:, orden]
    mediana = tabla["mediana_m"].to_numpy()[orden]
    p10 = tabla["p10_m"].to_numpy()[orden]
    p90 = tabla["p90_m"].to_numpy()[orden]
    n_validos = tabla["n_validos"].to_numpy()[orden]

    vector_derecha = signo * vector_medio
    izquierda = nombre_direccion(-vector_derecha)
    derecha = nombre_direccion(vector_derecha)

    bloque_izq = "Yaciente" if invertir else "Colgante"
    bloque_der = "Colgante" if invertir else "Yaciente"

    fig, ax = plt.subplots(figsize=(11, 5.5))

    if MOSTRAR_INDIVIDUALES:
        ax.plot(
            x, z.T,
            color="gray", linewidth=0.7, alpha=0.3
        )
        ax.plot(
            [], [], color="gray", linewidth=0.8,
            label="Perfiles individuales"
        )

    if MOSTRAR_BANDA:
        ax.fill_between(
            x, p10, p90,
            color="steelblue", alpha=0.2,
            label="80 % central de las elevaciones"
        )

    ax.plot(
        x, mediana,
        color="navy", linewidth=2.2,
        label="Mediana de los perfiles"
    )

    # ── Plano de falla esquemático ───────────────────────────────────
    MANTEO_SUPUESTO = 60.0
    PROFUNDIDAD_DIBUJADA_M = 100.0  # Solo controla la extensión visual

    # Anclar en la mediana del relieve, exactamente en la traza.
    indice_cero = int(np.argmin(np.abs(x)))
    z_traza = mediana[indice_cero]

    if np.isfinite(z_traza):
        # En la convención original, el colgante tiene distancia negativa.
        # "signo" incorpora la inversión geográfica de la figura.
        sentido_horizontal = -signo

        profundidad = np.linspace(0, PROFUNDIDAD_DIBUJADA_M, 50)
        desplazamiento = (
            profundidad / np.tan(np.radians(MANTEO_SUPUESTO))
        )

        x_plano = sentido_horizontal * desplazamiento
        z_plano = z_traza - profundidad

        ax.plot(
            x_plano, z_plano,
            color="black",
            linestyle="--",
            linewidth=2,
            label="Plano esquemático: manteo supuesto de 60°",
        )

        ax.scatter(
            [0], [z_traza],
            color="black", s=35, zorder=5,
            label="Traza en superficie",
        )

    else:
        # No inventar una elevación para anclar el plano.
        ax.text(
            0.5, 0.03,
            "Sin mediana válida en la traza: no se dibuja el plano.",
            transform=ax.transAxes,
            ha="center", fontsize=9, color="darkred",
        )

    # Marcar posiciones donde faltan perfiles válidos para la mediana.
    insuficiente = n_validos < MIN_PERFILES_VALIDOS
    ax.fill_between(
        x, 0, 1,
        where=insuficiente,
        transform=ax.get_xaxis_transform(),
        color="darkorange", alpha=0.12,
        label="Datos insuficientes para la mediana"
        if insuficiente.any() else None,
    )

    ax.text(
        0.01, 1.02,
        f"{izquierda}\n{bloque_izq}",
        transform=ax.transAxes,
        ha="left", va="bottom", fontweight="bold",
    )
    ax.text(
        0.99, 1.02,
        f"{derecha}\n{bloque_der}",
        transform=ax.transAxes,
        ha="right", va="bottom", fontweight="bold",
    )

    ax.set(
        xlim=(x.min(), x.max()),
        xlabel=(
            "Distancia transversal a la falla (m)\n"
            "0 = traza cartografiada; valores negativos a la izquierda"
        ),
        ylabel="Elevación del DEM (m)",
    )

    fig.suptitle(
        f"{r['falla']} — {clave}\n"
        f"{r['cinematica']} | {r['tipo']} | "
        f"Buzamiento hacia {r['dir_buz']}",
        y=1.04,
    )

    ax.grid(alpha=0.2)
    ax.legend(loc="best", fontsize=9)
    fig.tight_layout()

    return fig


for clave in SECTORES_VER:
    if clave not in resultados:
        raise KeyError(f"Sector inexistente: {clave}")

    fig = figura_sector(clave)
    plt.show()
    plt.close(fig)

## 6. Intersecciones entre cauces principales y fallas

Localizamos las intersecciones geométricas entre los cauces principales
y las trazas cartografiadas de las fallas.

Para cada punto registramos:

- Identificador y nombre del río.
- Identificador, nombre y cinemática de la falla.
- Coordenadas UTM.
- Distancia a lo largo del cauce desde el outlet.

Los ríos exportados están orientados desde aguas arriba hacia el outlet.
Por ello, la distancia desde el outlet se calcula restando la distancia
recorrida desde el inicio de la línea a su longitud total.

### Precauciones

- Una intersección cartográfica no demuestra actividad tectónica.
- Un contacto puntual puede ser tangencial: requiere revisión.
- Si río y falla comparten un tramo, se registra como superposición,
  no se transforma artificialmente en un único punto.
- No se aplica un buffer ni se desplazan las geometrías para forzar cruces.
- Un río sin intersecciones puede pasar cerca de una falla; la proximidad
  es un análisis distinto.
- Las distancias deben contrastarse con las tablas de perfiles antes
  de utilizarlas para marcar fallas sobre gráficos de χ o ksn.

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

RUTA_RIOS = Path(
    r"C:\datos\Memoria"
    r"\Codigos Stanford\Productos\Redes_de_drenaje"
    r"\20260929_144303_906995\rios_outlets.shp"
)

RUTA_FALLAS_CRUCES = Path(
    r"C:\datos\capas_externas\Nueva carpeta"
    r"\fallas_CHAF_area_estudio.shp"
)

# Leer nuevamente las capas: no depende de la selección de gráficos.
rios_cruces = gpd.read_file(RUTA_RIOS)
fallas_cruces = gpd.read_file(RUTA_FALLAS_CRUCES)

for nombre, capa in (
    ("Ríos", rios_cruces),
    ("Fallas", fallas_cruces),
):
    if capa.crs is None:
        raise ValueError(f"{nombre}: falta el sistema de coordenadas.")

    if (
        capa.geometry.isna().any()
        or capa.geometry.is_empty.any()
        or (~capa.geometry.is_valid).any()
    ):
        raise ValueError(f"{nombre}: hay geometrías vacías o inválidas.")

rios_cruces = rios_cruces.to_crs(epsg=32718)
fallas_cruces = fallas_cruces.to_crs(epsg=32718)

if not rios_cruces.geom_type.eq("LineString").all():
    raise ValueError(
        "Se esperaba una línea continua por río. "
        "Revisar antes de calcular distancias desde el outlet."
    )

if not fallas_cruces.geom_type.isin(
    ["LineString", "MultiLineString"]
).all():
    raise ValueError("La capa de fallas debe contener líneas.")

rios_cruces["rio_id"] = rios_cruces["rio_id"].astype(str)
fallas_cruces["f_id"] = fallas_cruces["f_id"].astype(str)

if rios_cruces["rio_id"].duplicated().any():
    raise ValueError("Hay IDs de río repetidos; revisar la capa.")

# IMPORTANTE:
# Esta convención corresponde a la celda con la que exportamos rios_outlets.
# No cambiarla a menos que hayas invertido las líneas posteriormente.
OUTLET_AL_FINAL = True


def separar_interseccion(geometria):
    """Separar puntos y tramos compartidos sin inventar cruces."""
    puntos, tramos = [], []

    if geometria.is_empty:
        return puntos, tramos

    if geometria.geom_type == "Point":
        puntos.append(geometria)

    elif geometria.geom_type == "LineString":
        tramos.append(geometria)

    elif hasattr(geometria, "geoms"):
        for parte in geometria.geoms:
            p, t = separar_interseccion(parte)
            puntos.extend(p)
            tramos.extend(t)

    else:
        raise ValueError(
            f"Intersección no esperada: {geometria.geom_type}"
        )

    return puntos, tramos


filas_cruces = []
filas_solapes = []

for _, rio in rios_cruces.iterrows():
    cauce = rio.geometry

    for registro_falla, (_, falla) in enumerate(
        fallas_cruces.iterrows(), start=1
    ):
        if not cauce.intersects(falla.geometry):
            continue

        interseccion = cauce.intersection(falla.geometry)
        puntos, tramos = separar_interseccion(interseccion)

        atributos = {
            "rio_id": rio["rio_id"],
            "rio": rio["rio"],
            "registro_falla": registro_falla,
            "f_id": falla["f_id"],
            "falla": falla["falla"],
            "tipo": falla.get("tipo", ""),
            "cinematica": falla.get("sentido", ""),
            "dir_buz": falla.get("dir_buz", ""),
        }

        for punto in puntos:
            distancia_inicio = cauce.project(punto)

            distancia_outlet = (
                cauce.length - distancia_inicio
                if OUTLET_AL_FINAL
                else distancia_inicio
            )

            filas_cruces.append({
                **atributos,
                "distancia_outlet_m": float(distancia_outlet),
                "x_utm": punto.x,
                "y_utm": punto.y,
                "geometry": punto,
            })

        for tramo in tramos:
            filas_solapes.append({
                **atributos,
                "longitud_compartida_m": tramo.length,
                "geometry": tramo,
            })

columnas_base = [
    "rio_id", "rio", "registro_falla", "f_id",
    "falla", "tipo", "cinematica", "dir_buz",
]

cruces = gpd.GeoDataFrame(
    filas_cruces,
    columns=columnas_base + [
        "distancia_outlet_m", "x_utm", "y_utm", "geometry"
    ],
    geometry="geometry",
    crs="EPSG:32718",
)

superposiciones = gpd.GeoDataFrame(
    filas_solapes,
    columns=columnas_base + [
        "longitud_compartida_m", "geometry"
    ],
    geometry="geometry",
    crs="EPSG:32718",
)

if not cruces.empty:
    cruces = cruces.sort_values(
        ["rio_id", "distancia_outlet_m"],
        key=lambda s: pd.to_numeric(s) if s.name == "rio_id" else s,
    ).reset_index(drop=True)

    cruces.insert(
        0, "cruce_id",
        [f"CR_{i:03d}" for i in range(1, len(cruces) + 1)]
    )

resumen_cruces = rios_cruces[["rio_id", "rio"]].copy()

conteos = cruces.groupby("rio_id").size()
conteos_solapes = superposiciones.groupby("rio_id").size()

resumen_cruces["n_intersecciones"] = (
    resumen_cruces["rio_id"].map(conteos).fillna(0).astype(int)
)
resumen_cruces["n_superposiciones"] = (
    resumen_cruces["rio_id"].map(conteos_solapes).fillna(0).astype(int)
)

print(f"Intersecciones puntuales: {len(cruces)}")
print(f"Tramos superpuestos: {len(superposiciones)}")
print(
    "Ríos con intersecciones puntuales:",
    int((resumen_cruces["n_intersecciones"] > 0).sum())
)

display(resumen_cruces)

if not cruces.empty:
    display(cruces.drop(columns="geometry"))

if not superposiciones.empty:
    print("Estos tramos requieren revisión por separado:")
    display(superposiciones.drop(columns="geometry"))

## 7. Ubicación de los cruces río–falla

Representamos las intersecciones sobre el DEM original:

- Azul: cauce principal.
- Rojo: trazas de fallas.
- Amarillo: intersecciones puntuales, identificadas por `cruce_id`.
- Magenta: tramos compartidos entre río y falla, si existen.

El norte está arriba y el este a la derecha.

La vista se concentra en los cruces de cada río. Las etiquetas indican
el identificador del punto y el nombre de la falla.

Los puntos representan intersecciones cartográficas, no evidencias
automáticas de actividad tectónica.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

from rasterio.windows import from_bounds, Window
from rasterio.enums import Resampling
from shapely.geometry import Point, LineString
from IPython.display import display

# ═════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN
# ═════════════════════════════════════════════════════════════════════

BASE_VIS = Path(
    r"C:\datos"
    r"\Memoria\Codigos Stanford"
)

RUTA_KP_VIS = BASE_VIS / "kp_manuales_28_09.csv"
RUTA_DEM_VIS = BASE_VIS.parent / "TDM_cut_UTM18.tif"
CARPETA_PERFILES_VIS = (
    BASE_VIS / "Productos" / "info_perfiles" / "20260928"
)

# Por defecto: ríos con cruces o tramos compartidos.
RIOS_VER_CRUCES = list(dict.fromkeys(
    cruces["rio_id"].astype(str).tolist()
    + superposiciones["rio_id"].astype(str).tolist()
))

# Alternativas:
# RIOS_VER_CRUCES = ["6", "10"]
# RIOS_VER_CRUCES = [str(i) for i in range(1, 19)]

MARGEN_M = 1500
MAX_PIXELES_MAPA = 1200

# "eventos": encuadre de cruces + KP.
# "rio_completo": encuadre del cauce completo.
ENCUADRE_MAPA = "eventos"

# None = perfil completo; ejemplo: (5, 15), en km desde el outlet.
LIMITES_PERFIL_KM = None

# Tolerancias de asociación, no errores cartográficos estimados.
MAX_DESAJUSTE_KP_M = 30.0
MAX_DESAJUSTE_CRUCE_M = 1.0

# ═════════════════════════════════════════════════════════════════════
# CARGAR KP Y REFERENCIAS DE LOS PERFILES
# ═════════════════════════════════════════════════════════════════════

kp_vis = pd.read_csv(
    RUTA_KP_VIS,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    dtype={"ID": str},
)
kp_vis["ID"] = kp_vis["ID"].str.strip()

for campo in ("x_utm", "y_utm", "distancia_m"):
    kp_vis[campo] = pd.to_numeric(kp_vis[campo], errors="raise")

manifest_vis = json.loads(
    (CARPETA_PERFILES_VIS / "manifest.json").read_text(
        encoding="utf-8"
    )
)

if ENCUADRE_MAPA not in ("eventos", "rio_completo"):
    raise ValueError("ENCUADRE_MAPA: 'eventos' o 'rio_completo'.")

for nombre, capa in (
    ("cruces", cruces),
    ("superposiciones", superposiciones),
    ("fallas", fallas_cruces),
):
    if capa.crs is None or capa.crs.to_epsg() != 32718:
        raise ValueError(f"{nombre}: se esperaba EPSG:32718.")

relaciones_vis = []
incidencias_vis = []


def valor_en_perfil(distancia, valores, posicion):
    """Interpolar sin atravesar intervalos con datos ausentes."""
    valores = np.asarray(valores, dtype=float)

    if posicion < distancia[0] or posicion > distancia[-1]:
        return np.nan

    j = int(np.searchsorted(distancia, posicion))

    if j < len(distancia) and np.isclose(
        distancia[j], posicion, atol=1e-6, rtol=0
    ):
        return valores[j]

    if j == 0 or j == len(distancia):
        return np.nan

    if not np.isfinite(valores[j-1:j+1]).all():
        return np.nan

    return float(np.interp(
        posicion,
        distancia[j-1:j+1],
        valores[j-1:j+1],
    ))


def dibujar_dem(ax, fig, limites):
    xmin, ymin, xmax, ymax = limites

    with rasterio.open(RUTA_DEM_VIS) as src:
        if src.crs is None or src.crs.to_epsg() != 32718:
            raise ValueError("El DEM debe estar en EPSG:32718.")

        izq = max(xmin, src.bounds.left)
        der = min(xmax, src.bounds.right)
        inf = max(ymin, src.bounds.bottom)
        sup = min(ymax, src.bounds.top)

        if izq >= der or inf >= sup:
            return

        ventana = from_bounds(
            izq, inf, der, sup, transform=src.transform
        )

        c0 = max(0, int(np.floor(ventana.col_off)))
        f0 = max(0, int(np.floor(ventana.row_off)))
        c1 = min(
            src.width,
            int(np.ceil(ventana.col_off + ventana.width))
        )
        f1 = min(
            src.height,
            int(np.ceil(ventana.row_off + ventana.height))
        )

        ventana = Window(c0, f0, c1-c0, f1-f0)
        factor = max(
            1,
            int(np.ceil(
                max(ventana.width, ventana.height) / MAX_PIXELES_MAPA
            ))
        )

        imagen = src.read(
            1,
            window=ventana,
            masked=True,
            out_shape=(
                max(1, int(ventana.height) // factor),
                max(1, int(ventana.width) // factor),
            ),
            resampling=Resampling.bilinear,
        )
        imagen = np.ma.masked_invalid(imagen)

        if imagen.count():
            izq, inf, der, sup = src.window_bounds(ventana)
            fondo = ax.imshow(
                imagen,
                extent=(izq, der, inf, sup),
                origin="upper",
                cmap="Greys_r",
                zorder=0,
            )
            fig.colorbar(
                fondo, ax=ax, shrink=0.55, pad=0.02,
                label="Elevación (m)"
            )


def mostrar_rio_integrado(rid):
    rid = str(rid)

    if rid not in manifest_vis["rios"]:
        raise KeyError(f"No existe perfil guardado para el río {rid}.")

    info = manifest_vis["rios"][rid]
    p = pd.read_csv(CARPETA_PERFILES_VIS / info["archivo"])

    distancia = p["distancia_m"].to_numpy(float)
    xy = p[["x_utm", "y_utm"]].to_numpy(float)

    if (
        len(p) < 2
        or not np.isfinite(xy).all()
        or not np.isfinite(distancia).all()
        or not np.all(np.diff(distancia) > 0)
        or not np.isclose(distancia[0], 0)
    ):
        raise ValueError(f"Río {rid}: revisar geometría y distancias.")

    distancia_xy = np.r_[
        0,
        np.cumsum(np.linalg.norm(np.diff(xy, axis=0), axis=1))
    ]

    if not np.allclose(
        distancia, distancia_xy, atol=0.1, rtol=0
    ):
        raise ValueError(
            f"Río {rid}: distancias y geometría no coinciden."
        )

    # El mapa usa exactamente el cauce del perfil guardado.
    cauce = LineString(xy)

    puntos = cruces[cruces["rio_id"].astype(str) == rid]
    solapes = superposiciones[
        superposiciones["rio_id"].astype(str) == rid
    ]

    eventos_falla = []
    eventos_kp = []

    # ── Verificar cruces ──────────────────────────────────────────────
    for _, cruce in puntos.iterrows():
        desajuste = cauce.distance(cruce.geometry)

        if desajuste > MAX_DESAJUSTE_CRUCE_M:
            incidencias_vis.append({
                "rio_id": rid,
                "elemento": cruce["cruce_id"],
                "motivo": "Cruce fuera del perfil guardado",
                "desajuste_m": desajuste,
            })
            continue

        eventos_falla.append({
            "etiqueta": cruce["cruce_id"],
            "falla": cruce["falla"],
            "distancia_m": float(cauce.project(cruce.geometry)),
            "geometry": cruce.geometry,
        })

    # ── Verificar KP ──────────────────────────────────────────────────
    for _, punto in kp_vis[kp_vis["ID"] == rid].iterrows():
        if not np.isfinite([punto["x_utm"], punto["y_utm"]]).all():
            incidencias_vis.append({
                "rio_id": rid,
                "elemento": punto["id_kp"],
                "motivo": "Coordenadas inválidas",
                "desajuste_m": np.nan,
            })
            continue

        geometria = Point(punto["x_utm"], punto["y_utm"])
        desajuste = cauce.distance(geometria)

        if desajuste > MAX_DESAJUSTE_KP_M:
            incidencias_vis.append({
                "rio_id": rid,
                "elemento": punto["id_kp"],
                "motivo": "KP alejado del perfil",
                "desajuste_m": desajuste,
            })
            continue

        eventos_kp.append({
            "id_kp": punto["id_kp"],
            "distancia_m": float(cauce.project(geometria)),
            "distancia_csv_m": punto["distancia_m"],
            "desajuste_m": desajuste,
            "geometry": geometria,
        })

    # Etiquetas ordenadas desde el outlet hacia aguas arriba.
    eventos_kp.sort(key=lambda e: e["distancia_m"])

    for numero, evento in enumerate(eventos_kp, start=1):
        evento["etiqueta"] = f"R{rid}_KP{numero}"

        cercano = (
            min(
                eventos_falla,
                key=lambda f: abs(
                    f["distancia_m"] - evento["distancia_m"]
                )
            )
            if eventos_falla else None
        )

        relaciones_vis.append({
            "rio_id": rid,
            "rio": info["nombre"],
            "etiqueta": evento["etiqueta"],
            "id_kp": evento["id_kp"],
            "distancia_kp_m": evento["distancia_m"],
            "diferencia_con_csv_m": (
                evento["distancia_m"] - evento["distancia_csv_m"]
            ),
            "desajuste_xy_m": evento["desajuste_m"],
            "cruce_cercano": cercano["etiqueta"] if cercano else None,
            "falla": cercano["falla"] if cercano else None,
            "separacion_por_cauce_m": (
                abs(evento["distancia_m"] - cercano["distancia_m"])
                if cercano else np.nan
            ),
        })

    # ── Encuadre: incluir tanto los cruces como los KP ─────────────────
    geometrias = [
        e["geometry"] for e in eventos_falla + eventos_kp
    ] + list(solapes.geometry)

    if ENCUADRE_MAPA == "rio_completo" or not geometrias:
        limites = np.asarray([cauce.bounds])
    else:
        limites = np.asarray([g.bounds for g in geometrias])

    xmin = limites[:, 0].min() - MARGEN_M
    ymin = limites[:, 1].min() - MARGEN_M
    xmax = limites[:, 2].max() + MARGEN_M
    ymax = limites[:, 3].max() + MARGEN_M

    # ── Una figura con tres paneles ───────────────────────────────────
    fig = plt.figure(figsize=(17, 9), constrained_layout=True)
    gs = fig.add_gridspec(
        2, 2, width_ratios=[1, 1.4], height_ratios=[1.6, 1]
    )

    ax_mapa = fig.add_subplot(gs[:, 0])
    ax_z = fig.add_subplot(gs[0, 1])
    ax_k = fig.add_subplot(gs[1, 1], sharex=ax_z)

    dibujar_dem(ax_mapa, fig, (xmin, ymin, xmax, ymax))

    fallas_vista = fallas_cruces.cx[xmin:xmax, ymin:ymax]
    if not fallas_vista.empty:
        fallas_vista.plot(
            ax=ax_mapa, color="firebrick", linewidth=1.5, zorder=2
        )

    ax_mapa.plot(
        xy[:, 0], xy[:, 1],
        color="royalblue", linewidth=1.5,
        label="Cauce del perfil", zorder=3,
    )

    if not solapes.empty:
        solapes.plot(
            ax=ax_mapa, color="magenta", linewidth=4, zorder=4
        )
        ax_mapa.plot(
            [], [], color="magenta", linewidth=4,
            label="Tramo compartido: revisar",
        )

    # ── Perfiles ──────────────────────────────────────────────────────
    ax_z.plot(
        distancia / 1000, p["elevacion_m"],
        color="steelblue", linewidth=1.5,
    )
    ax_k.plot(
        distancia / 1000, p["ksn"],
        color="slategray", linewidth=1.2,
    )

    for j, evento in enumerate(eventos_falla):
        punto = evento["geometry"]
        x = evento["distancia_m"] / 1000

        ax_mapa.scatter(
            punto.x, punto.y,
            marker="X", s=85, color="firebrick",
            edgecolor="white", linewidth=0.7,
            label="Cruce con falla" if j == 0 else None,
            zorder=6,
        )
        ax_mapa.annotate(
            evento["etiqueta"],
            (punto.x, punto.y),
            xytext=(7, 10), textcoords="offset points",
            fontsize=8, color="firebrick",
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"),
            zorder=7,
        )

        for ax in (ax_z, ax_k):
            ax.axvline(
                x, color="firebrick", linestyle="--",
                linewidth=1.2,
                label="Cruce con falla" if j == 0 else None,
            )

        ax_z.text(
            x, 0.98,
            f"{evento['etiqueta']} · {evento['falla']}",
            transform=ax_z.get_xaxis_transform(),
            rotation=90, ha="right", va="top",
            color="firebrick", fontsize=8,
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"),
            clip_on=True,
        )

    for j, evento in enumerate(eventos_kp):
        punto = evento["geometry"]
        posicion = evento["distancia_m"]
        x = posicion / 1000

        ax_mapa.scatter(
            punto.x, punto.y,
            marker="D", s=65, color="gold",
            edgecolor="black",
            label="KP manual" if j == 0 else None,
            zorder=6,
        )
        ax_mapa.annotate(
            evento["etiqueta"],
            (punto.x, punto.y),
            xytext=(7, -16), textcoords="offset points",
            fontsize=8,
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"),
            zorder=7,
        )

        for ax, campo in ((ax_z, "elevacion_m"), (ax_k, "ksn")):
            y = valor_en_perfil(distancia, p[campo], posicion)

            ax.axvline(
                x, color="darkorange", linestyle=":", linewidth=1,
                label="KP manual" if j == 0 else None,
            )

            if np.isfinite(y):
                ax.scatter(
                    x, y, marker="D", s=55,
                    color="gold", edgecolor="black", zorder=5,
                )

        # Etiqueta visible incluso si la elevación local es inválida.
        ax_z.text(
            x, 0.03, evento["etiqueta"],
            transform=ax_z.get_xaxis_transform(),
            rotation=90, ha="left", va="bottom", fontsize=8,
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"),
            clip_on=True,
        )

    # ── Orientación y formato ─────────────────────────────────────────
    ax_mapa.annotate(
        "N", xy=(0.94, 0.95), xytext=(0.94, 0.82),
        xycoords="axes fraction",
        ha="center", fontsize=13, fontweight="bold",
        arrowprops=dict(arrowstyle="-|>", color="black", lw=2),
    )

    ax_mapa.plot(
        [], [], color="firebrick", linewidth=1.5,
        label="Falla cartografiada",
    )
    ax_mapa.set(
        xlim=(xmin, xmax), ylim=(ymin, ymax),
        xlabel="Este UTM (m)", ylabel="Norte UTM (m)",
        title="Mapa · norte de cuadrícula arriba",
    )
    ax_mapa.set_aspect("equal")
    ax_mapa.ticklabel_format(style="plain", useOffset=False)
    ax_mapa.tick_params(axis="x", labelrotation=30)
    ax_mapa.legend(loc="lower left", fontsize=8)

    ax_z.set(
        title="Perfil longitudinal",
        ylabel="Elevación (m)",
    )
    ax_k.set(
        title="Pendiente normalizada",
        ylabel="ksn",
        xlabel=(
            "Distancia desde el outlet (km)\n"
            "Outlet a la izquierda · aguas arriba a la derecha"
        ),
    )

    for ax in (ax_z, ax_k):
        ax.grid(alpha=0.2)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(loc="best", fontsize=8)

    ax_k.set_xlim(
        *(LIMITES_PERFIL_KM if LIMITES_PERFIL_KM is not None
          else (0, distancia[-1] / 1000))
    )

    fig.suptitle(
        f"{rid} — {info['nombre']} | "
        f"{len(eventos_falla)} cruces y {len(eventos_kp)} KP asociados",
        fontsize=14,
    )

    plt.show()
    plt.close(fig)

    # Tabla para identificar los cruces de la figura.
    if eventos_falla:
        display(pd.DataFrame([
            {
                "cruce_id": e["etiqueta"],
                "falla": e["falla"],
                "distancia_outlet_m": e["distancia_m"],
            }
            for e in eventos_falla
        ]))


# ═════════════════════════════════════════════════════════════════════
# MOSTRAR Y CONSERVAR TABLAS EN MEMORIA
# ═════════════════════════════════════════════════════════════════════

for rid in dict.fromkeys(map(str, RIOS_VER_CRUCES)):
    mostrar_rio_integrado(rid)

tabla_relacion_kp_fallas = pd.DataFrame(relaciones_vis)
tabla_revision_asociaciones = pd.DataFrame(incidencias_vis)

if not tabla_relacion_kp_fallas.empty:
    print("Separación entre KP y cruce aceptado más cercano, por el cauce:")
    display(tabla_relacion_kp_fallas)

if not tabla_revision_asociaciones.empty:
    print("Elementos excluidos del mapa y los perfiles por desajuste:")
    display(tabla_revision_asociaciones)

## Perfiles longitudinales: geología, knickpoints, fallas y terrazas

Este análisis compara la posición de los knickpoints (KP) con las unidades geológicas, los contactos cartografiados, las fallas y las terrazas seleccionadas.

Se procesarán todos los ríos disponibles, pero inicialmente se mostrarán **Curanipe y Tregalemu**. Al incorporar nuevas cartas a la capa geológica, podremos actualizar el análisis sin cambiar su estructura.

### Qué mostrarán los gráficos

- **Perfil longitudinal:** elevación frente a distancia recorrida por el cauce desde el outlet, aumentando aguas arriba.
- **Unidades geológicas:** tramos del perfil coloreados según el código de cada polígono.
- **Knickpoints:** puntos del CSV. Si la selección del Excel es diferente, se mostrará por separado.
- **Contactos y fallas:** posiciones donde el cauce intersecta sus trazas.
- **Terrazas seleccionadas:** marcadores ubicados mediante su código en el catálogo de terrazas.

### Precauciones para interpretar los resultados

La clasificación geológica es preliminar. Las áreas sin cartografía se identificarán como **sin cobertura**, y las unidades sin resolver permanecerán como **pendientes**.

Las coordenadas del Excel corresponden a los KP, no a las terrazas. Para estas últimas se utilizarán las coordenadas del catálogo original. Su representación en el perfil será una **proyección lateral sobre el cauce**, conservando la elevación del marcador e indicando su separación al río. No representa necesariamente un cruce de la terraza con el cauce.

La falla Maule se mostrará únicamente como contexto, porque su traza generalizada no permite interpretar distancias métricas precisas.

La proximidad entre un KP y un contacto o una falla constituye una observación que debemos revisar; por sí sola no demuestra un control litológico o tectónico.

### Datos de entrada y formato requerido

Todos los archivos deben corresponder a la misma versión de los perfiles y selecciones. Las coordenadas de perfiles, KP y terrazas se expresan en **WGS 84 / UTM zona 18S (EPSG:32718)**, en metros.

Las capas vectoriales deben tener su CRS correctamente definido; si utilizan otro sistema, deberán reproyectarse antes del cruce espacial.

| Entrada | Formato | Campos necesarios | Convenciones |
|---|---|---|---|
| Perfiles longitudinales | Un CSV por río | `x_utm`, `y_utm`, `distancia_m`, `elevacion_m` | Coordenadas y elevaciones en metros. Distancia desde 0 en el outlet, aumentando aguas arriba. Filas ordenadas por distancia. |
| Catálogo de perfiles | `manifest.json` | `unidad_distancia`; objeto `rios` con ID, `nombre`, `archivo` y `outlet_xy` | `unidad_distancia` debe ser `"m"`. Cada archivo debe corresponder al ID indicado. |
| Knickpoints manuales | CSV | `ID`, `id_kp`, `indice`, `x_utm`, `y_utm`, `distancia_m`, `elevacion_m` | Separador `;` y coma decimal. `id_kp` único. `indice` comienza en 0 y corresponde a la fila del perfil original. |
| Unidades geológicas | Shapefile de polígonos | `codigo`; recomendados: `estado`, `unidad`, `nom_ref`, `color_hex` | Un código consistente por unidad. Los códigos ausentes o dudosos no deben tratarse como unidades confirmadas. |
| Fallas CHAF | Shapefile de líneas | Geometría y `falla` | Conservar también `f_id`, `tipo`, `sentido`, `rumbo`, `buz`, `dir_buz` y `refs` cuando estén disponibles. |
| Falla Maule preliminar | Shapefile de líneas | Geometría y `nombre` | Conservar `precision`, `uso` y `fuente`. Se utiliza únicamente como contexto. |
| Selección manual por río | Excel, hoja `Hoja1` | `id`, `Cauce`, `Terraza`, `MIS`, `ID knickpoint`, `Elevación (m)`, `Este UTM (m)`, `Norte UTM (m)` | Una fila por río. Los últimos cuatro campos corresponden al KP seleccionado, no a la terraza. |
| Catálogo de terrazas | CSV | `terrace_id`, `este_utm18s_m`, `norte_utm18s_m`, `elevacion_m`, `mis_asignado`, `crs_utm` | Separador `,` y punto decimal. Un registro por `terrace_id`. Proporciona la ubicación y elevación del marcador de terraza. |

#### Cómo se relacionan los archivos

| Relación | Identificador utilizado | Comprobación |
|---|---|---|
| Perfil ↔ KP | ID del río en `manifest.json` ↔ `ID` del CSV | El índice, las coordenadas, la distancia y la elevación del KP deben coincidir con el perfil. |
| Río ↔ selección manual | ID del río ↔ `id` del Excel | Utilizar el ID, no solamente el nombre del cauce. |
| KP del CSV ↔ KP seleccionado | `id_kp` ↔ `ID knickpoint` | Si difieren, conservar ambos y señalar la discrepancia. |
| Terraza seleccionada ↔ catálogo | `Terraza` ↔ `terrace_id` | Coincidencia exacta del código, por ejemplo `JM2015_163`. |
| Perfil ↔ geología y fallas | Intersección espacial | Geometrías válidas y sistemas de coordenadas compatibles. |

#### Reglas para actualizar y reproducir el análisis

1. **Conservar los encabezados indicados**, incluidos espacios y acentos.
2. Mantener los identificadores estables. No renumerar ríos, KP ni terrazas sin actualizar sus referencias.
3. Representar datos desconocidos mediante celdas vacías o valores nulos, **nunca mediante cero**. Si un río no tiene KP, dejar vacíos todos sus campos de KP.
4. Conservar juntos los archivos del shapefile: `.shp`, `.shx`, `.dbf`, `.prj` y, cuando exista, `.cpg`. El `.qml` es opcional y contiene el estilo.
5. Al incorporar nuevas cartas, mantener el campo `codigo`, documentar su fuente y revisar los solapamientos. No utilizar el mismo código para unidades diferentes sin distinguirlas.
6. Registrar las rutas y versiones utilizadas. Si cambian los perfiles, comprobar nuevamente la correspondencia de los KP.
7. Documentar cualquier corrección manual de geometrías, códigos o selecciones. La validez geométrica no equivale a una clasificación geológica confirmada.

**Nota:** el campo `Tasa Uplift` del Excel no se utiliza en esta comparación espacial. Su significado y unidad deben verificarse antes de incorporarlo a cálculos posteriores.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display


# ── 1. RUTAS ──────────────────────────────────────────────────────
geo_base = Path(
    r"C:\datos\Memoria"
)
geo_codigo = geo_base / "Codigos Stanford"

geo_rutas = {
    "perfiles": (
        geo_codigo / "Productos" / "info_perfiles" / "20260928"
    ),
    "kp": geo_codigo / "kp_manuales_28_09.csv",
    "geologia": Path(
        r"C:\datos\capas_externas\Chanco_unidades_clasificadas"
        r"\Chanco_unidades.shp"
    ),
    "chaf": Path(
        r"C:\datos\capas_externas\Nueva carpeta"
        r"\fallas_CHAF_area_estudio.shp"
    ),
    "maule": Path(
        r"C:\datos\capas_externas\Nueva carpeta"
        r"\falla_Maule_preliminar.shp"
    ),
    "seleccion": (
        geo_codigo / "Excel y csv" / "terrazas_kp_por_rio.xlsx"
    ),
    "catalogo": (
        geo_base / "shapes" / "Terrazas" / "terrazas_170_UTM18S.csv"
    ),
}

# CRS documentado de las coordenadas de perfiles, KP y terrazas.
geo_crs = "EPSG:32718"

# Se analizarán todos los ríos; esta lista controla las figuras futuras.
geo_rios_ver = ["10", "12"]  # Curanipe y Tregalemu
geo_avisos = []


# ── 2. FUNCIONES DE COMPROBACIÓN ───────────────────────────────────
def geo_texto(valor):
    return "" if pd.isna(valor) else str(valor).strip()


def geo_id(valor):
    numero = float(valor)
    if not np.isfinite(numero) or not numero.is_integer():
        raise ValueError(f"ID de río inválido: {valor}")
    return str(int(numero))


def geo_exigir_columnas(tabla, columnas, nombre):
    faltantes = set(columnas) - set(tabla.columns)
    if faltantes:
        raise ValueError(f"{nombre}: faltan columnas {sorted(faltantes)}")


def geo_leer_vector(ruta, tipos):
    capa = gpd.read_file(ruta)

    if capa.empty or capa.crs is None:
        raise ValueError(f"Capa vacía o sin CRS definido:\n{ruta}")

    if (
        capa.geometry.isna().any()
        or capa.geometry.is_empty.any()
        or not capa.geometry.is_valid.all()
        or not capa.geom_type.isin(tipos).all()
    ):
        raise ValueError(f"Revisar geometrías de:\n{ruta}")

    # Reproyección en memoria; no modifica el shapefile original.
    return capa.to_crs(geo_crs).reset_index(drop=True)


for geo_nombre, geo_ruta in geo_rutas.items():
    if not geo_ruta.exists():
        raise FileNotFoundError(f"No se encontró {geo_nombre}:\n{geo_ruta}")


# ── 3. GEOLOGÍA Y FALLAS ───────────────────────────────────────────
geo_pol = geo_leer_vector(
    geo_rutas["geologia"], ["Polygon", "MultiPolygon"]
)
geo_chaf = geo_leer_vector(
    geo_rutas["chaf"], ["LineString", "MultiLineString"]
)
geo_maule = geo_leer_vector(
    geo_rutas["maule"], ["LineString", "MultiLineString"]
)

geo_exigir_columnas(geo_pol, ["codigo"], "Geología")
geo_exigir_columnas(geo_chaf, ["falla"], "Fallas CHAF")
geo_exigir_columnas(geo_maule, ["nombre"], "Falla Maule")

for geo_columna in ["codigo", "estado", "unidad", "nom_ref", "color_hex"]:
    if geo_columna not in geo_pol:
        geo_pol[geo_columna] = ""
    geo_pol[geo_columna] = geo_pol[geo_columna].map(geo_texto)

geo_pol["codigo"] = geo_pol["codigo"].replace("", "SIN_ASIG")
geo_pol["estado"] = geo_pol["estado"].replace("", "No documentado")

geo_chaf["geo_fuente"] = "CHAF"
geo_chaf["geo_nombre"] = geo_chaf["falla"].map(geo_texto)
geo_chaf["geo_metrica"] = True

geo_maule["geo_fuente"] = "Maule"
geo_maule["geo_nombre"] = geo_maule["nombre"].map(geo_texto)
geo_maule["geo_metrica"] = False  # Solo contexto, no distancias métricas.

geo_fallas = gpd.GeoDataFrame(
    pd.concat([geo_chaf, geo_maule], ignore_index=True),
    geometry="geometry",
    crs=geo_crs,
)


# ── 4. KNICKPOINTS DEL CSV ─────────────────────────────────────────
geo_kp = pd.read_csv(
    geo_rutas["kp"], sep=";", decimal=","
).dropna(how="all")

geo_kp.columns = geo_kp.columns.str.strip()

geo_columnas_kp = [
    "ID", "id_kp", "indice", "x_utm", "y_utm",
    "distancia_m", "elevacion_m",
]
geo_exigir_columnas(geo_kp, geo_columnas_kp, "CSV de KP")

# El CSV puede incluir una fila de río sin KP seleccionado.
geo_sin_kp = geo_kp["id_kp"].map(geo_texto).eq("")
geo_campos_punto = [
    "indice", "x_utm", "y_utm", "distancia_m", "elevacion_m"
]

if geo_kp.loc[geo_sin_kp, geo_campos_punto].notna().any().any():
    raise ValueError("Hay datos de un KP sin id_kp. Revisar el CSV.")

for _, geo_fila in geo_kp.loc[geo_sin_kp].iterrows():
    geo_avisos.append(
        f"Sin KP seleccionado en CSV: {geo_texto(geo_fila.get('rio'))}"
    )

geo_kp = geo_kp.loc[~geo_sin_kp].copy()
geo_kp["ID"] = geo_kp["ID"].map(geo_id)
geo_kp["id_kp"] = geo_kp["id_kp"].map(geo_texto)

if geo_kp["id_kp"].duplicated().any():
    raise ValueError("El CSV contiene identificadores id_kp repetidos.")

for geo_columna in geo_campos_punto:
    geo_kp[geo_columna] = pd.to_numeric(
        geo_kp[geo_columna], errors="raise"
    )

if not np.isfinite(geo_kp[geo_campos_punto].to_numpy(float)).all():
    raise ValueError("Hay KP con coordenadas, índices o elevaciones ausentes.")


# ── 5. SELECCIÓN MANUAL Y CATÁLOGO DE TERRAZAS ──────────────────────
geo_sel = pd.read_excel(
    geo_rutas["seleccion"], sheet_name="Hoja1"
).dropna(how="all")

geo_sel.columns = geo_sel.columns.str.strip()

geo_exigir_columnas(
    geo_sel,
    [
        "id", "Cauce", "Terraza", "MIS", "ID knickpoint",
        "Elevación (m)", "Este UTM (m)", "Norte UTM (m)",
    ],
    "Excel de selección",
)

geo_sel["ID"] = geo_sel["id"].map(geo_id)

if geo_sel["ID"].duplicated().any():
    raise ValueError("El Excel debe contener una fila por río.")

geo_sel = geo_sel.set_index("ID")

geo_cat = pd.read_csv(geo_rutas["catalogo"])

geo_exigir_columnas(
    geo_cat,
    [
        "terrace_id", "este_utm18s_m", "norte_utm18s_m",
        "elevacion_m", "mis_asignado", "crs_utm",
    ],
    "Catálogo de terrazas",
)

geo_cat["terrace_id"] = geo_cat["terrace_id"].map(geo_texto)

if (
    geo_cat["terrace_id"].eq("").any()
    or geo_cat["terrace_id"].duplicated().any()
):
    raise ValueError("El catálogo tiene códigos de terraza vacíos o repetidos.")

if not geo_cat["crs_utm"].astype(str).str.contains("32718").all():
    raise ValueError("Verificar el CRS del catálogo de terrazas.")

geo_cat = geo_cat.set_index("terrace_id")


# ── 6. PERFILES YA EXPORTADOS: TODOS LOS RÍOS ──────────────────────
geo_archivo_manifest = geo_rutas["perfiles"] / "manifest.json"

geo_manifest = json.loads(
    geo_archivo_manifest.read_text(encoding="utf-8")
)

if geo_manifest.get("unidad_distancia") != "m":
    raise ValueError("Los perfiles deben tener distancias en metros.")

geo_perfiles = {}
geo_nombres = {}

for geo_rid, geo_meta in geo_manifest["rios"].items():
    geo_rid = geo_id(geo_rid)

    geo_p = pd.read_csv(
        geo_rutas["perfiles"] / geo_meta["archivo"]
    )

    geo_exigir_columnas(
        geo_p,
        ["x_utm", "y_utm", "distancia_m", "elevacion_m"],
        f"Perfil {geo_rid}",
    )

    geo_xy = geo_p[["x_utm", "y_utm"]].to_numpy(float)
    geo_d = geo_p["distancia_m"].to_numpy(float)

    if len(geo_p) < 2 or not np.isfinite(np.c_[geo_xy, geo_d]).all():
        raise ValueError(f"Río {geo_rid}: coordenadas o distancias inválidas.")

    geo_longitud_acumulada = np.r_[
        0.,
        np.cumsum(np.linalg.norm(np.diff(geo_xy, axis=0), axis=1)),
    ]

    if (
        np.any(np.diff(geo_d) <= 0)
        or not np.allclose(
            geo_d, geo_longitud_acumulada, atol=0.1, rtol=0
        )
    ):
        raise ValueError(
            f"Río {geo_rid}: distancia_m no coincide con la longitud "
            "del cauce desde el inicio del perfil."
        )

    geo_ajuste_outlet = np.linalg.norm(
        geo_xy[0] - np.asarray(geo_meta["outlet_xy"], dtype=float)
    )
    if geo_ajuste_outlet > 30:
        raise ValueError(
            f"Río {geo_rid}: revisar orientación y ajuste al outlet "
            f"({geo_ajuste_outlet:.1f} m)."
        )

    geo_perfiles[geo_rid] = geo_p
    geo_nombres[geo_rid] = geo_meta["nombre"]


# ── 7. COMPROBAR QUE LOS KP CORRESPONDEN A ESTOS PERFILES ───────────
geo_campos_control = [
    "x_utm", "y_utm", "distancia_m", "elevacion_m"
]

for _, geo_k in geo_kp.iterrows():
    geo_rid = geo_k["ID"]

    if geo_rid not in geo_perfiles:
        raise ValueError(f"KP de un río sin perfil cargado: {geo_rid}")

    geo_p = geo_perfiles[geo_rid]
    geo_indice = float(geo_k["indice"])

    if not geo_indice.is_integer() or not 0 <= geo_indice < len(geo_p):
        raise ValueError(f"Índice inválido: {geo_k['id_kp']}")

    if not np.allclose(
        geo_p.iloc[int(geo_indice)][geo_campos_control].to_numpy(float),
        geo_k[geo_campos_control].to_numpy(float),
        atol=0.2,
        rtol=0,
    ):
        raise ValueError(
            f"{geo_k['id_kp']}: el CSV y el perfil no coinciden. "
            "Revisar la carpeta de perfiles antes de continuar."
        )


# ── 8. RESUMEN Y DIFERENCIAS ENTRE CSV Y EXCEL ──────────────────────
geo_filas_revision = []

for geo_rid, geo_p in geo_perfiles.items():
    geo_ids_csv = geo_kp.loc[
        geo_kp["ID"].eq(geo_rid), "id_kp"
    ].tolist()

    geo_id_excel = ""
    geo_terraza = ""

    if geo_rid in geo_sel.index:
        geo_id_excel = geo_texto(geo_sel.loc[geo_rid, "ID knickpoint"])
        geo_terraza = geo_texto(geo_sel.loc[geo_rid, "Terraza"])
    else:
        geo_avisos.append(f"Río {geo_rid}: sin selección en el Excel.")

    if not geo_id_excel:
        geo_estado_revision = "Sin KP seleccionado en Excel"
    elif geo_id_excel in geo_ids_csv:
        geo_estado_revision = "Coincide"
    else:
        geo_estado_revision = "Difiere del CSV: revisar"

    if geo_terraza.startswith("JM2015_"):
        geo_estado_terraza = (
            "Encontrada en catálogo"
            if geo_terraza in geo_cat.index
            else "No encontrada: revisar"
        )
    else:
        geo_estado_terraza = "Sin terraza seleccionada"

    geo_filas_revision.append({
        "ID": geo_rid,
        "rio": geo_nombres[geo_rid],
        "puntos_perfil": len(geo_p),
        "KP_CSV": "; ".join(geo_ids_csv),
        "KP_Excel": geo_id_excel,
        "revision_KP": geo_estado_revision,
        "terraza": geo_terraza,
        "revision_terraza": geo_estado_terraza,
    })

geo_revision = pd.DataFrame(geo_filas_revision)

print(f"Perfiles cargados: {len(geo_perfiles)}")
print(f"KP cargados: {len(geo_kp)}")
print(f"Polígonos geológicos: {len(geo_pol)}")
print(f"Fallas CHAF: {len(geo_chaf)}")
print(f"Trazas Maule, solo contexto: {len(geo_maule)}")
print(f"CRS de trabajo: {geo_crs}")
print("\nNo se modificaron ni exportaron archivos.")

for geo_aviso in dict.fromkeys(geo_avisos):
    print("AVISO:", geo_aviso)

print("\nRevisión de todos los ríos:")
display(geo_revision)

print("\nCódigos y estados de la capa geológica:")
display(
    geo_pol.groupby(["codigo", "estado"])
    .size()
    .reset_index(name="poligonos")
)

## Cruces de los ríos con geología y fallas

Se analizarán todos los ríos y se mostrarán inicialmente Curanipe y Tregalemu.

- **KP:** se usarán exclusivamente los del CSV.
- **Geología:** se asignará una unidad a cada tramo y se localizarán los cambios de unidad.
- **Fallas:** se identificarán los cruces con CHAF; Maule se mostrará solo como contexto.
- **Terrazas:** se usarán las seleccionadas en el Excel, ubicadas mediante el catálogo original.

Las distancias KP–cruce se medirán siguiendo el cauce: **positivas aguas arriba y negativas aguas abajo**. Para las terrazas se indicará su separación lateral al río.

Las zonas sin cobertura, los solapamientos y las unidades pendientes se señalarán explícitamente. **Una coincidencia espacial no demuestra la causa del KP.**

Los resultados quedarán en memoria, sin exportar archivos.

In [ ]:
from shapely.geometry import Point, LineString


# ── 1. FUNCIONES ──────────────────────────────────────────────────
def geo_partes(geometria):
    """Extraer puntos o líneas de una intersección."""
    if geometria.is_empty:
        return
    if hasattr(geometria, "geoms"):
        for parte in geometria.geoms:
            yield from geo_partes(parte)
    else:
        yield geometria


def geo_posiciones(linea, interseccion):
    """Distancias sobre el cauce de puntos y extremos de intervalos."""
    posiciones = []

    for parte in geo_partes(interseccion):
        if parte.geom_type == "Point":
            posiciones.append(linea.project(parte))

        elif parte.geom_type in ("LineString", "LinearRing"):
            posiciones.extend([
                linea.project(Point(parte.coords[0])),
                linea.project(Point(parte.coords[-1])),
            ])

    return posiciones


def geo_unidad_en(punto):
    """No escoger arbitrariamente una unidad en límites o solapes."""
    indices = list(
        geo_pol.sindex.query(punto, predicate="intersects")
    )

    if not indices:
        return "SIN_COBERTURA", "Sin cobertura"

    if len(indices) > 1:
        return "AMBIGUO", "Solape o límite compartido"

    fila = geo_pol.iloc[indices[0]]
    return fila["codigo"], fila["estado"]


def geo_unidad_pendiente(codigo, estado):
    return (
        codigo in ("SIN_ASIG", "Hf/Hl1")
        or "pendiente" in str(estado).lower()
        or estado == "No documentado"
    )


# ── 2. GEOMETRÍA DE LOS CAUCES ─────────────────────────────────────
geo_lineas = {}

for geo_rid, geo_p in geo_perfiles.items():
    geo_linea = LineString(
        geo_p[["x_utm", "y_utm"]].to_numpy(float)
    )

    if not geo_linea.is_simple:
        raise ValueError(
            f"Río {geo_rid}: el cauce se intersecta consigo mismo. "
            "Revisar antes de proyectar puntos."
        )

    geo_lineas[geo_rid] = geo_linea


# ── 3. TRAMOS GEOLÓGICOS Y CONTACTOS ───────────────────────────────
geo_filas_tramos = []
geo_filas_contactos = []
geo_filas_cobertura = []

for geo_rid, geo_linea in geo_lineas.items():
    geo_cortes = [0.0, geo_linea.length]

    for geo_i in geo_pol.sindex.query(
        geo_linea, predicate="intersects"
    ):
        geo_interseccion = geo_linea.intersection(
            geo_pol.geometry.iloc[geo_i].boundary
        )
        geo_cortes.extend(
            geo_posiciones(geo_linea, geo_interseccion)
        )

    # Redondeo numérico de 1 mm; NO representa precisión cartográfica.
    geo_cortes = np.unique(
        np.clip(
            np.round(geo_cortes, 3),
            0.0,
            geo_linea.length,
        )
    )
    geo_cortes[0] = 0.0
    geo_cortes[-1] = geo_linea.length

    geo_tramos_rio = []

    for geo_a, geo_b in zip(geo_cortes[:-1], geo_cortes[1:]):
        if geo_b - geo_a <= 0:
            continue

        geo_medio = geo_linea.interpolate((geo_a + geo_b) / 2)
        geo_unidad, geo_estado = geo_unidad_en(geo_medio)

        geo_tramos_rio.append({
            "ID": geo_rid,
            "rio": geo_nombres[geo_rid],
            "desde_m": geo_a,
            "hasta_m": geo_b,
            "longitud_m": geo_b - geo_a,
            "codigo": geo_unidad,
            "estado": geo_estado,
        })

    geo_filas_tramos.extend(geo_tramos_rio)

    for geo_izq, geo_der in zip(
        geo_tramos_rio[:-1], geo_tramos_rio[1:]
    ):
        geo_c1 = geo_izq["codigo"]
        geo_c2 = geo_der["codigo"]

        if "SIN_COBERTURA" in (geo_c1, geo_c2):
            geo_tipo = "Borde de cobertura"

        elif "AMBIGUO" in (geo_c1, geo_c2):
            geo_tipo = "Límite ambiguo"

        elif geo_c1 == geo_c2:
            geo_tipo = "Límite mismo código"

        elif (
            geo_unidad_pendiente(geo_c1, geo_izq["estado"])
            or geo_unidad_pendiente(geo_c2, geo_der["estado"])
        ):
            geo_tipo = "Contacto pendiente"

        else:
            geo_tipo = "Contacto preliminar"

        geo_s = geo_izq["hasta_m"]
        geo_punto = geo_linea.interpolate(geo_s)

        geo_filas_contactos.append({
            "ID": geo_rid,
            "s_m": geo_s,
            "tipo": geo_tipo,
            "aguas_abajo": geo_c1,
            "aguas_arriba": geo_c2,
            "x_utm": geo_punto.x,
            "y_utm": geo_punto.y,
        })

    geo_sin_cobertura = sum(
        t["longitud_m"] for t in geo_tramos_rio
        if t["codigo"] == "SIN_COBERTURA"
    )
    geo_ambiguo = sum(
        t["longitud_m"] for t in geo_tramos_rio
        if t["codigo"] == "AMBIGUO"
    )
    geo_pendiente = sum(
        t["longitud_m"] for t in geo_tramos_rio
        if geo_unidad_pendiente(t["codigo"], t["estado"])
    )

    geo_filas_cobertura.append({
        "ID": geo_rid,
        "rio": geo_nombres[geo_rid],
        "longitud_km": geo_linea.length / 1000,
        "cobertura_pct": 100 * (
            1 - geo_sin_cobertura / geo_linea.length
        ),
        "sin_cobertura_km": geo_sin_cobertura / 1000,
        "clasificacion_pendiente_km": geo_pendiente / 1000,
        "ambiguo_km": geo_ambiguo / 1000,
    })

geo_tramos = pd.DataFrame(geo_filas_tramos)

geo_contactos = pd.DataFrame(
    geo_filas_contactos,
    columns=[
        "ID", "s_m", "tipo", "aguas_abajo",
        "aguas_arriba", "x_utm", "y_utm",
    ],
)

geo_cobertura = pd.DataFrame(geo_filas_cobertura)


# ── 4. INTERSECCIONES CON FALLAS ───────────────────────────────────
geo_filas_fallas = []

for geo_rid, geo_linea in geo_lineas.items():
    for geo_i in geo_fallas.sindex.query(
        geo_linea, predicate="intersects"
    ):
        geo_f = geo_fallas.iloc[geo_i]
        geo_interseccion = geo_linea.intersection(geo_f.geometry)

        for geo_parte in geo_partes(geo_interseccion):
            geo_pos = geo_posiciones(geo_linea, geo_parte)

            if not geo_pos:
                continue

            geo_filas_fallas.append({
                "ID": geo_rid,
                "traza_idx": int(geo_i),
                "falla": geo_f["geo_nombre"],
                "fuente": geo_f["geo_fuente"],
                "uso_metrico": bool(geo_f["geo_metrica"]),
                "tipo": (
                    "Puntual"
                    if geo_parte.geom_type == "Point"
                    else "Coincidencia lineal"
                ),
                "desde_m": min(geo_pos),
                "hasta_m": max(geo_pos),
            })

geo_cruces_fallas = pd.DataFrame(
    geo_filas_fallas,
    columns=[
        "ID", "traza_idx", "falla", "fuente", "uso_metrico",
        "tipo", "desde_m", "hasta_m",
    ],
).drop_duplicates()


# ── 5. TERRAZAS SELECCIONADAS EN EL EXCEL ──────────────────────────
# NO se utilizan aquí las coordenadas de KP del Excel.
geo_filas_terrazas = []
geo_avisos_analisis = []

for geo_rid, geo_linea in geo_lineas.items():
    if geo_rid not in geo_sel.index:
        continue

    geo_seleccion = geo_sel.loc[geo_rid]
    geo_tid = geo_texto(geo_seleccion["Terraza"])

    if not geo_tid.startswith("JM2015_"):
        continue

    if geo_tid not in geo_cat.index:
        geo_avisos_analisis.append(
            f"Río {geo_rid}: {geo_tid} no aparece en el catálogo."
        )
        continue

    geo_t = geo_cat.loc[geo_tid]

    geo_valores = pd.to_numeric(
        geo_t[[
            "este_utm18s_m", "norte_utm18s_m", "elevacion_m"
        ]],
        errors="coerce",
    ).to_numpy(float)

    if not np.isfinite(geo_valores).all():
        geo_avisos_analisis.append(
            f"{geo_tid}: faltan coordenadas o elevación."
        )
        continue

    geo_punto = Point(geo_valores[:2])
    geo_s = geo_linea.project(geo_punto)

    geo_filas_terrazas.append({
        "ID": geo_rid,
        "terraza": geo_tid,
        "s_m": geo_s,
        "elevacion_m": geo_valores[2],
        "x_utm": geo_punto.x,
        "y_utm": geo_punto.y,
        "separacion_lateral_m": geo_punto.distance(geo_linea),
        "proyeccion_en_extremo": (
            geo_s < 0.01 or geo_linea.length - geo_s < 0.01
        ),
        "MIS_excel": geo_texto(geo_seleccion["MIS"]),
        "MIS_catalogo": geo_texto(geo_t["mis_asignado"]),
    })

geo_terrazas = pd.DataFrame(
    geo_filas_terrazas,
    columns=[
        "ID", "terraza", "s_m", "elevacion_m",
        "x_utm", "y_utm", "separacion_lateral_m",
        "proyeccion_en_extremo", "MIS_excel", "MIS_catalogo",
    ],
)


# ── 6. TABLA DE COMPARACIÓN POR KP: SOLO CSV ───────────────────────
geo_filas_kp = []

for _, geo_k in geo_kp.iterrows():
    geo_rid = geo_k["ID"]
    geo_s = float(geo_k["distancia_m"])
    geo_punto = Point(geo_k["x_utm"], geo_k["y_utm"])

    geo_unidad, geo_estado = geo_unidad_en(geo_punto)

    geo_fila = {
        "ID": geo_rid,
        "rio": geo_nombres[geo_rid],
        "id_kp": geo_k["id_kp"],
        "kp_km": geo_s / 1000,
        "elevacion_m": geo_k["elevacion_m"],
        "x_utm": geo_k["x_utm"],
        "y_utm": geo_k["y_utm"],
        "unidad_kp": geo_unidad,
        "estado_unidad": geo_estado,
        "contacto_cercano": "",
        "tipo_contacto": "",
        "contacto_km": np.nan,
        "delta_contacto_m": np.nan,
        "cruce_CHAF": "",
        "cruce_CHAF_km": np.nan,
        "delta_cruce_CHAF_m": np.nan,
        "falla_CHAF_cercana": "",
        "distancia_recta_CHAF_m": np.nan,
        "terraza_seleccionada": "",
    }

    # Contacto más próximo siguiendo ESTE cauce.
    # Se excluyen bordes de cobertura y límites del mismo código.
    geo_candidatos = geo_contactos[
        geo_contactos["ID"].eq(geo_rid)
        & geo_contactos["tipo"].isin([
            "Contacto preliminar", "Contacto pendiente"
        ])
    ]

    if not geo_candidatos.empty:
        geo_indice = (
            geo_candidatos["s_m"] - geo_s
        ).abs().idxmin()
        geo_c = geo_candidatos.loc[geo_indice]

        geo_fila.update({
            "contacto_cercano": (
                f"{geo_c['aguas_abajo']} → {geo_c['aguas_arriba']}"
            ),
            "tipo_contacto": geo_c["tipo"],
            "contacto_km": geo_c["s_m"] / 1000,
            "delta_contacto_m": geo_c["s_m"] - geo_s,
        })

    # Cruce puntual CHAF más próximo siguiendo ESTE cauce.
    # Las coincidencias lineales quedan en su tabla como intervalos.
    geo_candidatos = geo_cruces_fallas[
        geo_cruces_fallas["ID"].eq(geo_rid)
        & geo_cruces_fallas["uso_metrico"].eq(True)
        & geo_cruces_fallas["tipo"].eq("Puntual")
    ]

    if not geo_candidatos.empty:
        geo_indice = (
            geo_candidatos["desde_m"] - geo_s
        ).abs().idxmin()
        geo_f = geo_candidatos.loc[geo_indice]

        geo_fila.update({
            "cruce_CHAF": geo_f["falla"],
            "cruce_CHAF_km": geo_f["desde_m"] / 1000,
            "delta_cruce_CHAF_m": geo_f["desde_m"] - geo_s,
        })

    # Distancia recta a CHAF: medida distinta de la distancia por cauce.
    geo_distancias = geo_chaf.geometry.distance(geo_punto)
    geo_indice = geo_distancias.idxmin()

    geo_fila.update({
        "falla_CHAF_cercana": geo_chaf.loc[
            geo_indice, "geo_nombre"
        ],
        "distancia_recta_CHAF_m": geo_distancias.loc[geo_indice],
    })

    if geo_rid in geo_sel.index:
        geo_fila["terraza_seleccionada"] = geo_texto(
            geo_sel.loc[geo_rid, "Terraza"]
        )

    geo_filas_kp.append(geo_fila)

geo_tabla_kp = pd.DataFrame(geo_filas_kp)


# ── 7. RESULTADOS EN PANTALLA ──────────────────────────────────────
print("Cobertura cartográfica de TODOS los ríos:")
print("La cobertura incluye polígonos de clasificación pendiente.")
display(geo_cobertura.round(2))

print("\nKP de Curanipe y Tregalemu: exclusivamente del CSV.")
print("Δs positivo: aguas arriba del KP; negativo: aguas abajo.")
print("NaN: sin cruce puntual evaluable; NO significa distancia cero.")

display(
    geo_tabla_kp[
        geo_tabla_kp["ID"].isin(geo_rios_ver)
    ].round(1)
)

print("\nTerrazas seleccionadas y separación lateral al río:")
display(
    geo_terrazas[
        geo_terrazas["ID"].isin(geo_rios_ver)
    ].round(1)
)

print("\nIntersecciones con fallas de los ríos seleccionados:")
display(
    geo_cruces_fallas[
        geo_cruces_fallas["ID"].isin(geo_rios_ver)
    ].round(1)
)

for geo_aviso in dict.fromkeys(geo_avisos_analisis):
    print("AVISO:", geo_aviso)

print(
    "\nAnálisis en memoria. No se guardaron archivos.\n"
    "Siguiente paso: gráficos longitudinales."
)

## Visualización de los perfiles

Se muestran inicialmente Curanipe y Tregalemu, con distancia desde el outlet aumentando aguas arriba.

- **Perfil coloreado:** unidad geológica de cada tramo.
- **Círculos rojos:** KP del CSV.
- **Líneas verticales punteadas:** contactos; naranja cuando la clasificación está pendiente.
- **Triángulos verdes:** terrazas proyectadas sobre el cauce, conservando su elevación e indicando la separación lateral.
- **Gris:** sectores sin cobertura cartográfica.

Las fallas solo se marcarán cuando intersecten el cauce. Las terrazas proyectadas no representan necesariamente un cruce real con el río.

**La figura permite comparar posiciones; no demuestra el origen de los KP.**

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import is_color_like
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


# ── VISUALIZACIÓN ─────────────────────────────────────────────────
geo_rios_ver = ["10", "12"]  # Curanipe y Tregalemu

# Límites del eje χ, en metros. Vacío = perfil completo.
# Ejemplo: geo_limites_chi_m = {"10": (0, 4000)}
geo_limites_chi_m = {}

geo_mostrar_terrazas = True
geo_mostrar_contactos = True
geo_mostrar_fallas = True


# ── COLORES ───────────────────────────────────────────────────────
geo_colores = {
    "SIN_COBERTURA": "#bdbdbd",
    "AMBIGUO": "#e979c9",
    "SIN_ASIG": "#888888",
}

for geo_n, geo_codigo_unidad in enumerate(
    sorted(geo_pol["codigo"].unique())
):
    if geo_codigo_unidad in geo_colores:
        continue

    geo_opciones = [
        color
        for color in geo_pol.loc[
            geo_pol["codigo"].eq(geo_codigo_unidad), "color_hex"
        ]
        if is_color_like(color)
    ]

    geo_colores[geo_codigo_unidad] = (
        geo_opciones[0]
        if geo_opciones
        else plt.get_cmap("tab20")(geo_n % 20)
    )


# ── PERFIL χ–ELEVACIÓN ────────────────────────────────────────────
def geo_figura_chi(rid):
    rid = geo_id(rid)
    p = geo_perfiles[rid]

    if "chi_m" not in p.columns:
        raise ValueError(
            f"Río {rid}: el perfil no contiene chi_m. "
            "Revisar la exportación de Perfiles_chi_ksn."
        )

    distancia = p["distancia_m"].to_numpy(dtype=float)
    chi = pd.to_numeric(
        p["chi_m"], errors="coerce"
    ).to_numpy(dtype=float)

    elevacion = pd.to_numeric(
        p["elevacion_m"], errors="coerce"
    ).to_numpy(dtype=float, copy=True)

    if (
        not np.isfinite(chi).all()
        or not np.isfinite(distancia).all()
        or np.any(np.diff(chi) <= 0)
        or np.any(np.diff(distancia) <= 0)
    ):
        raise ValueError(
            f"Río {rid}: χ y distancia deben ser finitos "
            "y estrictamente crecientes."
        )

    if "elevacion_valida" in p.columns:
        validos = (
            p["elevacion_valida"]
            .astype(str).str.lower()
            .isin(["true", "1", "1.0"])
            .to_numpy()
        )
        elevacion[~validos] = np.nan

    elevacion[~np.isfinite(elevacion)] = np.nan

    def a_chi(posiciones_m):
        """Posición por el cauce -> χ del mismo perfil."""
        valores = np.asarray(posiciones_m, dtype=float)

        if (
            not np.isfinite(valores).all()
            or np.any(valores < distancia[0] - 0.01)
            or np.any(valores > distancia[-1] + 0.01)
        ):
            raise ValueError(
                f"Río {rid}: posición fuera del perfil o inválida."
            )

        return np.interp(valores, distancia, chi)

    xmin, xmax = geo_limites_chi_m.get(
        rid, (float(chi[0]), float(chi[-1]))
    )

    if not chi[0] <= xmin < xmax <= chi[-1]:
        raise ValueError(
            f"Río {rid}: límites de χ fuera del intervalo "
            f"{chi[0]:.1f}–{chi[-1]:.1f} m."
        )

    # Convertir el encuadre χ a distancia solo para filtrar registros.
    smin, smax = np.interp([xmin, xmax], chi, distancia)

    tramos = geo_tramos[
        geo_tramos["ID"].eq(rid)
        & (geo_tramos["hasta_m"] >= smin)
        & (geo_tramos["desde_m"] <= smax)
    ]

    contactos = geo_contactos[
        geo_contactos["ID"].eq(rid)
        & geo_contactos["s_m"].between(smin, smax)
        & geo_contactos["tipo"].isin([
            "Contacto preliminar", "Contacto pendiente"
        ])
    ]

    fallas = geo_cruces_fallas[
        geo_cruces_fallas["ID"].eq(rid)
        & (geo_cruces_fallas["hasta_m"] >= smin)
        & (geo_cruces_fallas["desde_m"] <= smax)
    ]

    kp = geo_kp[
        geo_kp["ID"].eq(rid)
        & geo_kp["distancia_m"].between(smin, smax)
    ]

    terrazas = geo_terrazas[
        geo_terrazas["ID"].eq(rid)
        & geo_terrazas["s_m"].between(smin, smax)
    ]

    fig, (ax, banda) = plt.subplots(
        2, 1,
        figsize=(14, 7.5),
        sharex=True,
        gridspec_kw={
            "height_ratios": [7, 0.7],
            "hspace": 0.08,
        },
    )

    ax.plot(
        chi, elevacion,
        color="#333333", linewidth=4, zorder=2,
    )

    unidades_visibles = []
    hay_pendientes = False

    # Perfil y banda geológica: ambos en coordenadas χ.
    for t in tramos.itertuples():
        puntos = np.r_[
            t.desde_m,
            distancia[
                (distancia > t.desde_m)
                & (distancia < t.hasta_m)
            ],
            t.hasta_m,
        ]

        alturas = np.interp(puntos, distancia, elevacion)
        color = geo_colores[t.codigo]
        pendiente = geo_unidad_pendiente(t.codigo, t.estado)

        ax.plot(
            a_chi(puntos), alturas,
            color=color, linewidth=2.7, zorder=3,
        )

        banda.axvspan(
            float(a_chi(t.desde_m)),
            float(a_chi(t.hasta_m)),
            facecolor=color,
            edgecolor="#555555",
            linewidth=0.35,
            hatch=(
                "xxx" if t.codigo == "SIN_COBERTURA"
                else "///" if pendiente
                else None
            ),
        )

        if t.codigo not in unidades_visibles:
            unidades_visibles.append(t.codigo)

        hay_pendientes |= pendiente

    # Contactos.
    if geo_mostrar_contactos:
        for c in contactos.itertuples():
            color = (
                "#e58b25"
                if c.tipo == "Contacto pendiente"
                else "#8055a5"
            )

            ax.axvline(
                float(a_chi(c.s_m)),
                color=color, linestyle=":",
                linewidth=1.1, alpha=0.8, zorder=1,
            )

    # Fallas.
    if geo_mostrar_fallas:
        for n, f in enumerate(fallas.itertuples()):
            color = (
                "#222222" if f.fuente == "CHAF" else "#287bb5"
            )
            x0 = float(a_chi(f.desde_m))
            x1 = float(a_chi(f.hasta_m))

            if f.tipo == "Puntual":
                ax.axvline(
                    x0, color=color, linestyle="--",
                    linewidth=1.2, zorder=1,
                )
            else:
                ax.axvspan(
                    x0, x1, color=color, alpha=0.15, zorder=1,
                )

            etiqueta = f.falla
            if f.fuente == "Maule":
                etiqueta += " · contexto"

            ax.text(
                (max(x0, xmin) + min(x1, xmax)) / 2,
                0.98 - 0.13 * (n % 3),
                etiqueta,
                transform=ax.get_xaxis_transform(),
                rotation=90, va="top", ha="right",
                fontsize=8, color=color,
            )

    # KP: exclusivamente los del CSV.
    for n, k in enumerate(kp.itertuples(), start=1):
        x = float(a_chi(k.distancia_m))
        izquierda = x > xmin + 0.75 * (xmax - xmin)

        ax.scatter(
            x, k.elevacion_m,
            s=85, facecolor="#cc2438",
            edgecolor="white", linewidth=1, zorder=6,
        )

        ax.annotate(
            f"KP {n}\nχ = {x:.1f} m\n"
            f"Elevación = {k.elevacion_m:.1f} m",
            xy=(x, k.elevacion_m),
            xytext=(-12 if izquierda else 12, 22),
            textcoords="offset points",
            ha="right" if izquierda else "left",
            fontsize=9, color="#ad1930",
            arrowprops={
                "arrowstyle": "-",
                "color": "#ad1930",
                "linewidth": 0.8,
            },
        )

    # Terrazas: χ de su proyección al río y elevación del marcador.
    if geo_mostrar_terrazas:
        for n, t in enumerate(terrazas.itertuples()):
            x = float(a_chi(t.s_m))
            izquierda = x > xmin + 0.75 * (xmax - xmin)

            ax.scatter(
                x, t.elevacion_m,
                s=120, marker="^",
                facecolor="#187747",
                edgecolor="white", linewidth=1, zorder=6,
            )

            etiqueta = (
                f"{t.terraza} · {t.MIS_excel}\n"
                f"Elevación: {t.elevacion_m:.1f} m\n"
                f"Separación lateral: "
                f"{t.separacion_lateral_m / 1000:.2f} km"
            )

            if t.proyeccion_en_extremo:
                etiqueta += "\nProyección en extremo del perfil"
            if t.MIS_excel != t.MIS_catalogo:
                etiqueta += "\nMIS distinto del catálogo: revisar"

            ax.annotate(
                etiqueta,
                xy=(x, t.elevacion_m),
                xytext=(-15 if izquierda else 15, 48 + 15 * n),
                textcoords="offset points",
                ha="right" if izquierda else "left",
                fontsize=8.5, color="#14633b",
                arrowprops={
                    "arrowstyle": "-",
                    "color": "#187747",
                    "linewidth": 0.8,
                },
            )

    # Ajustar elevaciones al encuadre.
    alturas = np.r_[
        elevacion[(chi >= xmin) & (chi <= xmax)],
        np.interp([smin, smax], distancia, elevacion),
        kp["elevacion_m"].to_numpy(float),
    ]

    if geo_mostrar_terrazas:
        alturas = np.r_[
            alturas, terrazas["elevacion_m"].to_numpy(float)
        ]

    alturas = alturas[np.isfinite(alturas)]

    if len(alturas):
        zmin, zmax = alturas.min(), alturas.max()
        margen = max((zmax - zmin) * 0.25, 15)
        ax.set_ylim(zmin - margen * 0.4, zmax + margen)

    cobertura = geo_cobertura.loc[
        geo_cobertura["ID"].eq(rid), "cobertura_pct"
    ].iloc[0]

    parametros = (
        geo_manifest["rios"].get(rid, {}).get("parametros", {})
    )
    theta = parametros.get("theta_ref", "no documentado")
    a0 = parametros.get("A0_m2", "no documentado")

    ax.set_title(
        f"{geo_nombres[rid]} · Perfil χ–elevación\n"
        f"θ = {theta} · A₀ = {a0} m² · "
        f"Cobertura del cauce: {cobertura:.1f}%",
        fontsize=12, pad=14,
    )
    ax.set_ylabel("Elevación (m)")
    ax.grid(alpha=0.15)

    banda.set_xlim(xmin, xmax)
    banda.set_ylim(0, 1)
    banda.set_yticks([])
    banda.set_ylabel("Geología", rotation=0, labelpad=35)
    banda.set_xlabel("χ (m) → aguas arriba")

    # Leyenda.
    leyenda = []

    if not kp.empty:
        leyenda.append(Line2D(
            [], [], marker="o", linestyle="none",
            markerfacecolor="#cc2438", markeredgecolor="white",
            markersize=8, label="KP del CSV",
        ))

    if geo_mostrar_terrazas and not terrazas.empty:
        leyenda.append(Line2D(
            [], [], marker="^", linestyle="none",
            markerfacecolor="#187747", markeredgecolor="white",
            markersize=9, label="Terraza proyectada",
        ))

    if geo_mostrar_contactos:
        for tipo, color in [
            ("Contacto preliminar", "#8055a5"),
            ("Contacto pendiente", "#e58b25"),
        ]:
            if contactos["tipo"].eq(tipo).any():
                leyenda.append(Line2D(
                    [], [], color=color, linestyle=":", label=tipo
                ))

    if geo_mostrar_fallas:
        for fuente, color in [
            ("CHAF", "#222222"), ("Maule", "#287bb5")
        ]:
            if fallas["fuente"].eq(fuente).any():
                leyenda.append(Line2D(
                    [], [], color=color, linestyle="--",
                    label=fuente if fuente == "CHAF" else "Maule: contexto",
                ))

        leyenda.extend([
        Patch(
            facecolor=geo_colores[codigo],
            edgecolor="#555555",
            hatch="xxx" if codigo == "SIN_COBERTURA" else None,
            label="Sin cobertura" if codigo == "SIN_COBERTURA" else codigo,
        )
        for codigo in unidades_visibles
    ])

    if hay_pendientes:
        leyenda.append(Patch(
            facecolor="white", edgecolor="#555555",
            hatch="///", label="Clasificación pendiente",
        ))

    filas_leyenda = max(1, int(np.ceil(len(leyenda) / 5)))
    fig.subplots_adjust(
        left=0.09, right=0.97, top=0.86,
        bottom=0.16 + 0.035 * filas_leyenda,
    )
    fig.legend(
        handles=leyenda,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.015),
        ncol=5, fontsize=8.5, frameon=False,
    )

    return fig


# ── MOSTRAR, SIN EXPORTAR ─────────────────────────────────────────
for geo_rid in geo_rios_ver:
    geo_fig = geo_figura_chi(geo_rid)
    plt.show()
    plt.close(geo_fig)